# Agent Reflection in Pharmacogenomics
## LangChain vs. Direct Anthropic SDK

Welcome! In this lab you will implement the **reflection pattern** for a real clinical task: pharmacogenomics (PGx) drug recommendation.

Given a patient's genetic **diplotype** (pair of gene alleles), an LLM first drafts a clinical recommendation from its training knowledge alone (V1). The agent then consults an authoritative CPIC guideline database, critiques its own answer against that data, and produces a refined, evidence-grounded recommendation (V2).

You will build this workflow **twice** — once with **LangChain** (LCEL chains) and once with the **Anthropic SDK** directly — so you can compare the abstractions and choose the right tool for your own projects.

## 1. Introduction

### 1.1. Lab overview

The workflow you will implement:

1. **Generate V1** — LLM drafts a recommendation from memory (no database consulted).  
2. **Look up CPIC guideline** — fetch the authoritative gene × drug × phenotype record.  
3. **Reflect** — LLM critiques V1 against the guideline and identifies gaps.  
4. **Generate V2** — produce a precise, CPIC-grounded recommendation.

### What is CPIC?

[CPIC (Clinical Pharmacogenomics Implementation Consortium)](https://cpicpgx.org) publishes peer-reviewed guidelines that translate a patient's genetic test results into actionable drug-therapy decisions.

The lookup chain is:

```
Diplotype  →  Metabolizer phenotype  →  Clinical recommendation
(e.g., CYP2D6 *4/*4)  →  (Poor Metabolizer)  →  (Avoid codeine)
```

The CPIC database used in this lab was originally set up following the workflow described in the [PGxQA project](https://github.com/KarlKeat/PGxQA).

### 🎯 1.2. Learning outcomes

By the end of this lab you will be able to:

* Apply the **reflection pattern** to a clinical NLP task.
* Build a **LangChain** reflection pipeline using LCEL (`|` operator, `ChatPromptTemplate`, `StrOutputParser`).
* Build the **same pipeline** with the **Anthropic SDK** directly — no framework abstractions.
* Use **external feedback** (authoritative database records) as the grounding signal for reflection.
* Compare both approaches and reason about the trade-offs.

## 2. Setup: Initialize environment and clients

Import the standard libraries and define a small `show()` helper that renders styled output in the notebook (analogous to `utils.print_html` in other labs).

In [21]:
import json
import re
from IPython.display import HTML, Markdown, display
from html import escape
from dotenv import load_dotenv

_ = load_dotenv()

DARK_BOX_STYLE = """
    border:1px solid #374151;
    background:#111827;
    color:#e5e7eb;
    border-radius:6px;
    padding:10px;
    margin:6px 0;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
"""

DARK_TITLE_BOX_STYLE = """
    border:1px solid #3b82f6;
    border-left:6px solid #60a5fa;
    background:#111827;
    color:#e5e7eb;
    border-radius:8px;
    padding:12px 14px;
    margin:8px 0;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
"""

DARK_PRE_STYLE = """
    margin:0;
    white-space:pre-wrap;
    font-size:0.9em;
    line-height:1.45;
    color:#e5e7eb;
    background:#0f172a;
    border-radius:6px;
    padding:10px;
    overflow-x:auto;
"""

def show(content, title: str | None = None, diff: bool = False) -> None:
    """Render content as a styled dark-theme box in the notebook."""
    if diff:
        display(Markdown(str(content)))
        return

    escaped_content = escape(str(content))

    if title:
        escaped_title = escape(title)
        html = f"""
        <div style="{DARK_TITLE_BOX_STYLE}">
            <strong style="color:#93c5fd;">{escaped_title}</strong>
            <pre style="{DARK_PRE_STYLE}; margin-top:8px;">{escaped_content}</pre>
        </div>
        """
    else:
        html = f"""
        <div style="{DARK_BOX_STYLE}">
            <pre style="{DARK_PRE_STYLE}">{escaped_content}</pre>
        </div>
        """

    display(HTML(html))

### 2.1. Initialize clients and choose models

We initialize both clients here so both workflow sections can reuse them without re-importing.

In [17]:
import anthropic
from langchain_anthropic import ChatAnthropic

# ── Anthropic SDK client (Section 4) ────────────────────────────────────────
sdk_client = anthropic.Anthropic()

# ── Model choices ────────────────────────────────────────────────────────────
# Use a fast model for the initial draft and a more capable model for reflection.
# Feel free to swap these out — any Claude model will work.
GENERATION_MODEL = "claude-haiku-4-5-20251001"   # fast, cost-effective for V1
REFLECTION_MODEL = "claude-sonnet-4-6"             # stronger reasoning for reflection

print("Clients ready.")
print(f"  Generation model : {GENERATION_MODEL}")
print(f"  Reflection model : {REFLECTION_MODEL}")

Clients ready.
  Generation model : claude-haiku-4-5-20251001
  Reflection model : claude-sonnet-4-6


### 2.2. Load the CPIC mini-dataset

In a production system this data would be fetched from the CPIC PostgreSQL database (see [first_time_setup_part2.sh](https://github.com/KarlKeat/PGxQA/blob/main/first_time_setup_part2.sh)). For this lab we use a curated in-memory snapshot covering three clinically important gene–drug pairs from the [PGxQA benchmark](https://github.com/KarlKeat/PGxQA).

Each entry records the metabolizer **phenotype**, the clinical **implication**, the CPIC **recommendation**, and a short **classification** label.

In [18]:
# ── Diplotype → Metabolizer phenotype ──────────────────────────────────────
DIPLOTYPE_TO_PHENOTYPE = {
    "CYP2D6": {
        "*1/*1":   "Normal Metabolizer",
        "*1/*2":   "Normal Metabolizer",
        "*1/*4":   "Intermediate Metabolizer",
        "*1/*5":   "Intermediate Metabolizer",
        "*4/*4":   "Poor Metabolizer",
        "*4/*5":   "Poor Metabolizer",
        "*5/*5":   "Poor Metabolizer",
        "*1/*1xN": "Ultrarapid Metabolizer",
        "*2/*2xN": "Ultrarapid Metabolizer",
    },
    "CYP2C19": {
        "*1/*1":  "Normal Metabolizer",
        "*1/*2":  "Intermediate Metabolizer",
        "*1/*3":  "Intermediate Metabolizer",
        "*2/*2":  "Poor Metabolizer",
        "*2/*3":  "Poor Metabolizer",
        "*1/*17": "Rapid Metabolizer",
        "*17/*17":"Ultrarapid Metabolizer",
    },
    "TPMT": {
        "*1/*1":  "Normal Metabolizer",
        "*1/*2":  "Intermediate Metabolizer",
        "*1/*3A": "Intermediate Metabolizer",
        "*1/*3C": "Intermediate Metabolizer",
        "*3A/*3A":"Poor Metabolizer",
        "*3A/*3C":"Poor Metabolizer",
        "*3C/*3C":"Poor Metabolizer",
    },
}

# ── Gene × Drug × Phenotype → Clinical guideline ───────────────────────────
CPIC_GUIDELINES = {
    ("CYP2D6", "codeine"): {
        "Normal Metabolizer": {
            "implication": "Normal morphine formation from codeine.",
            "recommendation": "Use label-recommended age- or weight-specific dosing.",
            "classification": "No action needed",
        },
        "Intermediate Metabolizer": {
            "implication": (
                "Reduced morphine formation following codeine administration; "
                "reduced analgesic effect anticipated."
            ),
            "recommendation": (
                "Use label-recommended dosing. If there is no response and opioid use "
                "is still warranted, consider a non-codeine opioid."
            ),
            "classification": "Use with caution",
        },
        "Poor Metabolizer": {
            "implication": (
                "Greatly reduced morphine formation; endogenous morphine likely absent. "
                "Insufficient pain relief expected."
            ),
            "recommendation": (
                "Avoid codeine. Tramadol also not recommended. Consider a non-opioid "
                "analgesic or an opioid not predominantly metabolized by CYP2D6 "
                "(e.g., morphine, hydromorphone, oxymorphone)."
            ),
            "classification": "Avoid",
        },
        "Ultrarapid Metabolizer": {
            "implication": (
                "Markedly increased morphine formation; risk of serious or "
                "life-threatening toxicity, including respiratory depression."
            ),
            "recommendation": (
                "Avoid codeine due to risk of opioid toxicity. If opioid analgesia "
                "is needed, select an agent not predominantly metabolized by CYP2D6."
            ),
            "classification": "Avoid",
        },
    },
    ("CYP2C19", "clopidogrel"): {
        "Normal Metabolizer": {
            "implication": "Normal clopidogrel metabolism and antiplatelet effect.",
            "recommendation": "Use label-recommended dosing.",
            "classification": "No action needed",
        },
        "Intermediate Metabolizer": {
            "implication": (
                "Reduced platelet inhibition; increased residual platelet aggregation; "
                "elevated risk of cardiovascular events."
            ),
            "recommendation": (
                "Consider a 2× standard maintenance dose of clopidogrel, or switch to "
                "prasugrel or ticagrelor if clinically appropriate."
            ),
            "classification": "Use with caution",
        },
        "Poor Metabolizer": {
            "implication": (
                "Significantly reduced platelet inhibition; greatly elevated risk of "
                "major adverse cardiovascular events."
            ),
            "recommendation": (
                "Avoid clopidogrel. Use an alternative antiplatelet agent such as "
                "prasugrel or ticagrelor."
            ),
            "classification": "Avoid",
        },
        "Rapid Metabolizer": {
            "implication": "Normal-to-increased clopidogrel activation; no adverse signal.",
            "recommendation": "Use label-recommended dosing.",
            "classification": "No action needed",
        },
        "Ultrarapid Metabolizer": {
            "implication": "Normal-to-increased clopidogrel activation; no adverse signal.",
            "recommendation": "Use label-recommended dosing.",
            "classification": "No action needed",
        },
    },
    ("TPMT", "azathioprine"): {
        "Normal Metabolizer": {
            "implication": "Normal thiopurine metabolism; standard myelosuppression risk.",
            "recommendation": "Use label-recommended dosing.",
            "classification": "No action needed",
        },
        "Intermediate Metabolizer": {
            "implication": "Elevated risk of myelosuppression from azathioprine.",
            "recommendation": (
                "Start at 30–70 % of the standard dose; adjust based on tolerance and efficacy."
            ),
            "classification": "Alter dose",
        },
        "Poor Metabolizer": {
            "implication": (
                "Greatly elevated risk of severe, life-threatening myelosuppression."
            ),
            "recommendation": (
                "Reduce starting dose by 10-fold; administer no more than 3× per week. "
                "If treating a myelosuppressive disease, consider a non-thiopurine "
                "immunosuppressant."
            ),
            "classification": "Alter dose or avoid",
        },
    },
}

total_diplotypes = sum(len(v) for v in DIPLOTYPE_TO_PHENOTYPE.values())
total_guidelines = sum(len(v) for v in CPIC_GUIDELINES.values())
print(f"Loaded {total_diplotypes} diplotype–phenotype mappings across "
      f"{len(DIPLOTYPE_TO_PHENOTYPE)} genes")
print(f"Loaded {total_guidelines} gene–drug–phenotype guidelines across "
      f"{len(CPIC_GUIDELINES)} gene–drug pairs")

Loaded 23 diplotype–phenotype mappings across 3 genes
Loaded 12 gene–drug–phenotype guidelines across 3 gene–drug pairs


### 2.3. Lookup helper functions

These three functions encapsulate the database query logic. The LangChain and SDK workflows both call the same helpers — only the LLM interaction differs between the two implementations.

In [19]:
def lookup_phenotype(gene: str, diplotype: str) -> str | None:
    """Return the metabolizer phenotype for a gene diplotype."""
    gene_map = DIPLOTYPE_TO_PHENOTYPE.get(gene.upper())
    if not gene_map:
        return None
    key = diplotype.strip()
    if key in gene_map:
        return gene_map[key]
    # Try reversed allele order (e.g. *4/*1 → *1/*4)
    parts = key.split("/")
    if len(parts) == 2:
        return gene_map.get(f"{parts[1]}/{parts[0]}")
    return None


def lookup_cpic_guideline(gene: str, drug: str, phenotype: str) -> dict | None:
    """Return the CPIC recommendation dict for a gene–drug–phenotype combination."""
    drug_map = CPIC_GUIDELINES.get((gene.upper(), drug.lower()))
    if not drug_map:
        return None
    return drug_map.get(phenotype)


def format_guideline(gene: str, diplotype: str, drug: str,
                     phenotype: str, g: dict) -> str:
    """Format a CPIC lookup result as a readable multi-line string."""
    return (
        f"Gene:            {gene}\n"
        f"Diplotype:       {diplotype}\n"
        f"Phenotype:       {phenotype}\n"
        f"Drug:            {drug}\n"
        f"{'─'*50}\n"
        f"Implication:     {g['implication']}\n"
        f"Recommendation:  {g['recommendation']}\n"
        f"Classification:  {g['classification']}"
    )

Try the lookup helpers with the example we'll use throughout the lab.

In [22]:
gene, diplotype, drug = "CYP2D6", "*4/*4", "codeine"

phenotype = lookup_phenotype(gene, diplotype)
guideline = lookup_cpic_guideline(gene, drug, phenotype)

show(format_guideline(gene, diplotype, drug, phenotype, guideline),
     "CPIC Guideline Lookup")

---
## 3. LangChain Implementation

LangChain provides the **LangChain Expression Language (LCEL)**, which lets you compose prompts, models, and parsers with the `|` operator into readable chains:

```python
chain = prompt | llm | parser
result = chain.invoke({"key": "value"})
```

This declarative style makes it easy to swap components (e.g., change the model) without touching the control flow.

### 3.1. Step 1 — Generate the initial recommendation (V1)

The first step asks the LLM for a recommendation using **only its training knowledge**. No CPIC database is consulted at this stage.

The function below builds a two-message prompt (system + human), feeds it into a `ChatAnthropic` model, and parses the response as plain text with `StrOutputParser`.

In [23]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def generate_v1_langchain(gene: str, diplotype: str, drug: str,
                           model: str) -> str:
    """
    Draft a PGx recommendation from LLM memory using a LangChain LCEL chain.
    No CPIC database is consulted — this is the uninformed first draft.
    """
    llm = ChatAnthropic(model=model)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a clinical pharmacogenomics expert. "
         "Respond in 3–5 sentences: state the likely metabolizer phenotype, "
         "the clinical implication for the drug, and your recommendation."),
        ("human",
         "A patient carries the {gene} diplotype {diplotype}. "
         "What is the clinical guidance regarding {drug} therapy?"),
    ])

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"gene": gene, "diplotype": diplotype, "drug": drug})

In [24]:
v1_lc = generate_v1_langchain("CYP2D6", "*4/*4", "codeine", GENERATION_MODEL)
show(v1_lc, f"Step 1 — Initial Recommendation V1 [{GENERATION_MODEL}]")

### 3.2. Step 2 — Look up the CPIC guideline (external feedback)

This is the **execution step** — equivalent to running an SQL query and observing its output. Instead of leaving the LLM to guess, we fetch the authoritative CPIC record for this gene–drug–phenotype combination and pass it to the reflection step.

In [25]:
# Perform the CPIC lookup (uses the helpers defined in Section 2.3)
phenotype = lookup_phenotype("CYP2D6", "*4/*4")
guideline  = lookup_cpic_guideline("CYP2D6", "codeine", phenotype)

show(format_guideline("CYP2D6", "*4/*4", "codeine", phenotype, guideline),
     "Step 2 — CPIC Guideline (External Feedback)")

### 3.3. Step 3 — Reflect on V1 and produce a refined recommendation (V2)

The reflection step presents the LLM with:
- The original clinical question and the V1 recommendation
- The authoritative CPIC guideline retrieved in Step 2

The model is instructed to evaluate the accuracy of V1 and return a **JSON object** containing a short critique (`feedback`) and a refined recommendation (`refined_recommendation`).

In [26]:
def reflect_and_refine_langchain(
    gene: str,
    diplotype: str,
    drug: str,
    phenotype: str,
    v1: str,
    guideline: dict,
    model: str,
) -> tuple[str, str, str]:
    """
    Critique V1 against the CPIC guideline and return a refined recommendation.
    Uses a LangChain LCEL chain; output is parsed from JSON.

    Returns:
        feedback              – 1–3 sentence critique of V1
        refined_recommendation – CPIC-grounded V2 recommendation
        classification         – one of the CPIC action categories
    """
    llm = ChatAnthropic(model=model)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a clinical pharmacogenomics reviewer. "
         "Compare the initial recommendation against the authoritative CPIC guideline. "
         "Return ONLY a valid JSON object with three keys: "
         '\"feedback\" (1–3 sentences: what was correct / missing / incorrect in V1), '
         '\"refined_recommendation\" (3–5 sentences grounded in the CPIC data), '
         '\"classification\" (exactly one of: '
         '\"No action needed\", \"Use with caution\", \"Alter dose\", \"Avoid\"). '
         "No markdown, no extra text."),
        ("human",
         "Gene: {gene}  |  Diplotype: {diplotype}  |  Drug: {drug}  |  Phenotype: {phenotype}\n\n"
         "Initial recommendation (V1):\n{v1}\n\n"
         "Authoritative CPIC guideline:\n"
         "  Implication:     {implication}\n"
         "  Recommendation:  {cpic_rec}\n"
         "  Classification:  {cpic_class}\n\n"
         "Return the JSON review."),
    ])

    chain = prompt | llm | StrOutputParser()
    raw = chain.invoke({
        "gene": gene, "diplotype": diplotype,
        "drug": drug, "phenotype": phenotype,
        "v1": v1,
        "implication": guideline["implication"],
        "cpic_rec":    guideline["recommendation"],
        "cpic_class":  guideline["classification"],
    })

    # ── Parse JSON (with fallback) ─────────────────────────────────────────
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{[\s\S]*?\}", raw)
        obj = json.loads(m.group(0)) if m else {}

    return (
        obj.get("feedback", ""),
        obj.get("refined_recommendation", v1),
        obj.get("classification", guideline["classification"]),
    )

In [27]:
feedback_lc, v2_lc, cls_lc = reflect_and_refine_langchain(
    gene="CYP2D6", diplotype="*4/*4", drug="codeine",
    phenotype=phenotype,
    v1=v1_lc,
    guideline=guideline,
    model=REFLECTION_MODEL,
)

show(feedback_lc, "Step 3 — Reflection Feedback")
show(v2_lc,       f"Step 3 — Refined Recommendation V2 [{REFLECTION_MODEL}]")
show(f"CPIC Classification: {cls_lc}", "Final Classification")

### 3.4. Put it all together — the LangChain workflow

The `run_pgx_workflow_langchain` function chains Steps 1–3 into a single callable end-to-end pipeline.

In [11]:
def run_pgx_workflow_langchain(
    gene: str,
    diplotype: str,
    drug: str,
    generation_model: str = GENERATION_MODEL,
    reflection_model: str = REFLECTION_MODEL,
) -> dict:
    """
    End-to-end PGx reflection workflow (LangChain).

    Steps:
      1) Generate V1 recommendation from LLM memory.
      2) Look up the authoritative CPIC guideline.
      3) Reflect on V1 → produce refined V2 recommendation.
    """
    show(f"Gene: {gene}  |  Diplotype: {diplotype}  |  Drug: {drug}", "Query")

    # ── Step 1: Generate V1 ────────────────────────────────────────────────
    show("Step 1: Generating initial recommendation (V1)…")
    v1 = generate_v1_langchain(gene, diplotype, drug, generation_model)
    show(v1, f"Step 1 — Initial Recommendation V1 [{generation_model}]")

    # ── Step 2: CPIC lookup ────────────────────────────────────────────────
    show("Step 2: Looking up CPIC guideline…")
    phenotype = lookup_phenotype(gene, diplotype)
    guideline  = lookup_cpic_guideline(gene, drug, phenotype) if phenotype else None

    if phenotype is None:
        show(f"Diplotype {diplotype!r} not found for {gene}.", "Step 2 — CPIC Lookup")
        return {"v1": v1, "v2": v1, "feedback": "No phenotype found."}
    if guideline is None:
        show(f"No CPIC guideline for {gene} × {drug}.", "Step 2 — CPIC Lookup")
        return {"v1": v1, "v2": v1, "feedback": "No guideline found."}

    show(format_guideline(gene, diplotype, drug, phenotype, guideline),
         "Step 2 — CPIC Guideline (External Feedback)")

    # ── Step 3: Reflect ────────────────────────────────────────────────────
    show("Step 3: Reflecting on V1 against CPIC guideline…")
    feedback, v2, cls = reflect_and_refine_langchain(
        gene, diplotype, drug, phenotype, v1, guideline, reflection_model,
    )
    show(feedback, "Step 3 — Reflection Feedback")
    show(v2,       f"Step 3 — Refined Recommendation V2 [{reflection_model}]")
    show(f"CPIC Classification: {cls}", "Final Classification")

    return {
        "gene": gene, "diplotype": diplotype, "drug": drug,
        "phenotype": phenotype,
        "v1": v1,
        "guideline": guideline,
        "feedback": feedback,
        "v2": v2,
        "classification": cls,
    }

### 3.5. Try the workflow

Run the full LangChain workflow on the lecture example. After that, try swapping the gene–drug combination to see how the reflection adapts.

In [28]:
result_lc = run_pgx_workflow_langchain(
    gene="CYP2D6",
    diplotype="*4/*4",
    drug="codeine",
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
)

**Experiment ideas** — try replacing the inputs above:

| Gene | Diplotype | Drug | Expected classification |
|------|-----------|------|------------------------|
| CYP2D6 | *1/*1xN | codeine | Avoid (ultrarapid → toxicity risk) |
| CYP2C19 | *2/*2 | clopidogrel | Avoid |
| CYP2C19 | *1/*17 | clopidogrel | No action needed |
| TPMT | *1/*3A | azathioprine | Alter dose |
| TPMT | *3A/*3A | azathioprine | Alter dose or avoid |

---
## 4. Direct Anthropic SDK Implementation

In this section you will re-implement the same three-step workflow **without** LangChain — using the `anthropic` Python SDK directly.

Instead of `ChatPromptTemplate | llm | StrOutputParser()`, you call `sdk_client.messages.create(...)` and handle the response yourself. The logic is identical; only the API surface differs.

### 4.1. Step 1 — Generate the initial recommendation (V1)

The `system` and `user` messages are passed directly to `messages.create`. The response text is extracted from `response.content[0].text`.

In [36]:
def generate_v1_sdk(gene: str, diplotype: str, drug: str, model: str) -> str:
    """
    Draft a PGx recommendation from LLM memory using the Anthropic SDK directly.
    No CPIC database consulted — uninformed first draft.
    """
    response = sdk_client.messages.create(
        model=model,
        max_tokens=512,
        system=(
            "You are a clinical pharmacogenomics expert. "
            "Respond in 3–5 sentences: state the likely metabolizer phenotype, "
            "the clinical implication for the drug, and your recommendation."
        ),
        messages=[{
            "role": "user",
            "content": (
                f"A patient carries the {gene} diplotype {diplotype}. "
                f"What is the clinical guidance regarding {drug} therapy?"
            ),
        }],
    )
    return response.content[0].text

In [ ]:
v1_sdk = generate_v1_sdk("CYP2D6", "*4/*4", "codeine", GENERATION_MODEL)
show(v1_sdk, f"Step 1 — Initial Recommendation V1 [{GENERATION_MODEL}]")

### 4.2. Step 2 — Look up the CPIC guideline (external feedback)

The lookup uses the exact same `lookup_phenotype` / `lookup_cpic_guideline` helpers from Section 2.3 — the database query layer is framework-independent.

### 4.3. Step 3 — Reflect on V1 and produce a refined recommendation (V2)

The prompt instructions are the same as in the LangChain version; the difference is that we construct the message list manually and call `sdk_client.messages.create` directly instead of invoking an LCEL chain.

In [38]:
def reflect_and_refine_sdk(
    gene: str,
    diplotype: str,
    drug: str,
    phenotype: str,
    v1: str,
    guideline: dict,
    model: str,
) -> tuple[str, str, str]:
    """
    Critique V1 against the CPIC guideline using the Anthropic SDK directly.

    Returns:
        feedback              – 1–3 sentence critique of V1
        refined_recommendation – CPIC-grounded V2 recommendation
        classification         – one of the CPIC action categories
    """
    user_content = (
        f"Gene: {gene}  |  Diplotype: {diplotype}  |  "
        f"Drug: {drug}  |  Phenotype: {phenotype}\n\n"
        f"Initial recommendation (V1):\n{v1}\n\n"
        f"Authoritative CPIC guideline:\n"
        f"  Implication:     {guideline['implication']}\n"
        f"  Recommendation:  {guideline['recommendation']}\n"
        f"  Classification:  {guideline['classification']}\n\n"
        "Return the JSON review."
    )

    response = sdk_client.messages.create(
        model=model,
        max_tokens=1024,
        system=(
            "You are a clinical pharmacogenomics reviewer. "
            "Compare the initial recommendation against the authoritative CPIC guideline. "
            "Return ONLY a valid JSON object with three keys: "
            '"feedback" (1–3 sentences: what was correct / missing / incorrect in V1), '
            '"refined_recommendation" (3–5 sentences grounded in the CPIC data), '
            '"classification" (exactly one of: '
            '"No action needed", "Use with caution", "Alter dose", "Avoid"). '
            "No markdown, no extra text."
        ),
        messages=[{"role": "user", "content": user_content}],
    )

    raw = response.content[0].text

    # ── Parse JSON (with fallback) ─────────────────────────────────────────
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{[\s\S]*?\}", raw)
        obj = json.loads(m.group(0)) if m else {}

    return (
        obj.get("feedback", ""),
        obj.get("refined_recommendation", v1),
        obj.get("classification", guideline["classification"]),
    )

### 4.4. Put it all together — the direct SDK workflow

Same structure as `run_pgx_workflow_langchain`; only the function calls in Steps 1 and 3 are different.

In [ ]:
def run_pgx_workflow_sdk(
    gene: str,
    diplotype: str,
    drug: str,
    generation_model: str = GENERATION_MODEL,
    reflection_model: str = REFLECTION_MODEL,
) -> dict:
    """
    End-to-end PGx reflection workflow (Anthropic SDK).

    Steps:
      1) Generate V1 recommendation from LLM memory.
      2) Look up the authoritative CPIC guideline.
      3) Reflect on V1 → produce refined V2 recommendation.
    """
    show(f"Gene: {gene}  |  Diplotype: {diplotype}  |  Drug: {drug}", "Query")

    # ── Step 1 ─────────────────────────────────────────────────────────────
    show("Step 1: Generating initial recommendation (V1)…")
    v1 = generate_v1_sdk(gene, diplotype, drug, generation_model)
    show(v1, f"Step 1 — Initial Recommendation V1 [{generation_model}]")

    # ── Step 2 ─────────────────────────────────────────────────────────────
    show("Step 2: Looking up CPIC guideline…")
    phenotype = lookup_phenotype(gene, diplotype)
    guideline  = lookup_cpic_guideline(gene, drug, phenotype) if phenotype else None

    if phenotype is None:
        show(f"Diplotype {diplotype!r} not found for {gene}.", "Step 2 — CPIC Lookup")
        return {"v1": v1, "v2": v1, "feedback": "No phenotype found."}
    if guideline is None:
        show(f"No CPIC guideline for {gene} × {drug}.", "Step 2 — CPIC Lookup")
        return {"v1": v1, "v2": v1, "feedback": "No guideline found."}

    show(format_guideline(gene, diplotype, drug, phenotype, guideline),
         "Step 2 — CPIC Guideline (External Feedback)")

    # ── Step 3 ─────────────────────────────────────────────────────────────
    show("Step 3: Reflecting on V1 against CPIC guideline…")
    feedback, v2, cls = reflect_and_refine_sdk(
        gene, diplotype, drug, phenotype, v1, guideline, reflection_model,
    )
    show(feedback, "Step 3 — Reflection Feedback")
    show(v2,       f"Step 3 — Refined Recommendation V2 [{reflection_model}]")
    show(f"CPIC Classification: {cls}", "Final Classification")

    return {
        "gene": gene, "diplotype": diplotype, "drug": drug,
        "phenotype": phenotype,
        "v1": v1,
        "guideline": guideline,
        "feedback": feedback,
        "v2": v2,
        "classification": cls,
    }

### 4.5. Try the workflow

In [ ]:
result_sdk = run_pgx_workflow_sdk(
    gene="CYP2D6",
    diplotype="*4/*4",
    drug="codeine",
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
)

---
## 5. Comparing the Two Approaches

Run the cell below to see the V1 → V2 diff for both implementations side by side, rendered using the `redlines` package (tracked-changes style).

In [ ]:
from redlines import Redlines

# Run both workflows on a new example (CYP2C19 Poor Metabolizer + clopidogrel)
result_lc2  = run_pgx_workflow_langchain("CYP2C19", "*2/*2", "clopidogrel")
result_sdk2 = run_pgx_workflow_sdk(      "CYP2C19", "*2/*2", "clopidogrel")

print("\n── LangChain: V1 → V2 diff ──────────────────────────────────────────")
show(Redlines(result_lc2["v1"], result_lc2["v2"]).output_markdown,
     title="LangChain V1 → V2 changes", diff=True)

print("\n── Direct SDK: V1 → V2 diff ─────────────────────────────────────────")
show(Redlines(result_sdk2["v1"], result_sdk2["v2"]).output_markdown,
     title="Direct SDK V1 → V2 changes", diff=True)

### When to choose each approach

| | **LangChain** | **Direct SDK** |
|---|---|---|
| **Boilerplate** | Less — chains handle prompt rendering | More — prompts constructed manually |
| **Composability** | High — swap models/parsers by changing one line | Lower — change requires refactoring |
| **Transparency** | Lower — framework layers hide the API call | Higher — every parameter is explicit |
| **Debugging** | Use `set_debug(True)` or LangSmith | Inspect `response` object directly |
| **Dependency** | Requires `langchain-core`, `langchain-anthropic` | Only `anthropic` |
| **Best for** | Rapid prototyping, multi-step pipelines | Production services, minimal dependencies |

---
## 6. Final Takeaways

In this lab you practiced applying the **reflection pattern** to a real clinical task and implemented it with two different API surfaces.

Key ideas to remember:

* **Reflection with external feedback** is more reliable than self-reflection alone.   Providing the authoritative CPIC guideline (equivalent to query execution results in   the SQL lab) grounds the refinement in verifiable data rather than the LLM's memory.

* **LangChain** shines for composable, swappable pipelines. Changing the model,   adding a retrieval step, or wiring in a vector store is a one-line change.

* **Direct SDK** is preferable when you want minimal dependencies, maximum transparency,   or a thin wrapper around a specific Claude feature (streaming, tool use, etc.).

* The **CPIC database** used here can be reconstructed from the full PostgreSQL dump by   following [KarlKeat/PGxQA](https://github.com/KarlKeat/PGxQA). The   `diplotype_to_guideline_tests.py` script in that repo auto-generates QA benchmarks   for every gene–drug–phenotype combination — a useful resource for evaluating   PGx-aware LLM applications.

<div style="border:1px solid #22c55e;border-left:6px solid #16a34a;background:#dcfce7;border-radius:6px;padding:14px 16px;color:#064e3b;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">

🎉 <strong>Congratulations!</strong>

You have built an end-to-end agentic reflection workflow for pharmacogenomics using <strong>both LangChain and the direct Anthropic SDK</strong>.

You are now ready to:
<ul>
  <li>Apply the reflection pattern to other clinical or scientific databases.</li>
  <li>Choose the right API surface for your project's needs.</li>
  <li>Extend the workflow with retrieval-augmented generation (RAG) or tool use.</li>
</ul>
</div>

---
## 7. Scaling Up: Full CPIC Database via the REST API

The mini-dataset in Section 2 covered three gene–drug pairs to keep the lab self-contained. Here you will **replace it with the complete CPIC guidelines** fetched live from the [public CPIC REST API](https://api.cpicpgx.org/v1/) — the same data that the [PGxQA setup script](https://github.com/KarlKeat/PGxQA/blob/main/first_time_setup_part2.sh) downloads as a PostgreSQL dump and that [`diplotype_to_guideline_tests.py`](https://github.com/KarlKeat/PGxQA/blob/main/scripts/question_generation/diplotype_to_guideline_tests.py) queries to build QA benchmarks — but accessed over HTTP with no local database required.

After loading the full data you will run the **same LangChain and SDK workflow functions** from Sections 3 and 4 on gene–drug pairs that are not present in the mini-dataset.

### 7.1. Fetch guidelines from the CPIC REST API

The API is a [PostgREST](https://postgrest.org) interface over the CPIC PostgreSQL database.

| Endpoint | Rows | Contents |
|----------|------|----------|
| `/drug` | ~100 | Drug names; maps `drugid` → `name` |
| `/recommendation` | 2,159 | Gene × drug × phenotype → clinical recommendation |
| `/diplotype` | 110,000+ | Gene × diplotype → phenotype (fetched per-gene, on demand) |

In [14]:
import requests
import pandas as pd

CPIC_API = "https://api.cpicpgx.org/v1"

def fetch_all_pages(endpoint: str, page_size: int = 2000, **params) -> list[dict]:
    """Paginate through a CPIC REST API endpoint and return all rows."""
    rows, offset = [], 0
    while True:
        resp = requests.get(
            f"{CPIC_API}/{endpoint}",
            params={"limit": page_size, "offset": offset, **params},
            timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json()
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return rows

In [29]:
# ── 1. Fetch drug names ────────────────────────────────────────────────────
print("Fetching drug names…")
drug_rows  = fetch_all_pages("drug", select="drugid,name")
drugid_to_name = {r["drugid"]: r["name"] for r in drug_rows}
print(f"  {len(drugid_to_name)} drugs loaded")

# ── 2. Fetch all recommendations ──────────────────────────────────────────
print("Fetching recommendations…")
rec_rows = fetch_all_pages(
    "recommendation",
    select=(
        "drugid,phenotypes,allelestatus,implications,"
        "drugrecommendation,classification,dosinginformation,alternatedrugavailable"
    ),
)
print(f"  {len(rec_rows)} recommendation rows fetched")

# ── 3. Build a tidy DataFrame ─────────────────────────────────────────────
def _derive_action(rec_text: str, alternate: bool, dosing: bool) -> str:
    """Infer an action category from recommendation text and boolean API flags."""
    t = rec_text.lower()
    if alternate or any(w in t for w in ["avoid", "contraindicated", "not recommended",
                                          "do not use", "is contraindicated"]):
        return "Avoid"
    if dosing or any(w in t for w in ["reduce", "lower", "adjust", "alter",
                                       "50%", "30%", "10-fold", "alternative dose",
                                       "start with", "specific dose"]):
        return "Alter dose"
    if "no recommendation" in t:
        return "No recommendation"
    return "No action needed"

rows = []
for r in rec_rows:
    drug_name = drugid_to_name.get(r["drugid"], r["drugid"])
    # Metabolizer genes use phenotypes{}; allele-status genes (HLA) use allelestatus{}
    lookup = r["phenotypes"] if r["phenotypes"] else r["allelestatus"]
    if not lookup:
        continue
    for gene, phenotype in lookup.items():
        rows.append({
            "gene":            gene,
            "drug":            drug_name,
            "phenotype":       phenotype,
            "implication":     r["implications"].get(gene, ""),
            "recommendation":  r["drugrecommendation"],
            "action":          _derive_action(
                                   r["drugrecommendation"],
                                   r["alternatedrugavailable"],
                                   r["dosinginformation"],
                               ),
            "strength":        r["classification"],  # CPIC evidence strength
        })

df_recs_full = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["gene", "drug", "phenotype"])   # same phenotype, different activity scores
    .reset_index(drop=True)
)

print(f"  {len(df_recs_full)} unique gene × drug × phenotype combinations")
print(f"  Genes: {sorted(df_recs_full['gene'].unique())}")

Fetching drug names…
  323 drugs loaded
Fetching recommendations…
  2159 recommendation rows fetched
  631 unique gene × drug × phenotype combinations
  Genes: ['ABCG2', 'CACNA1S', 'CFTR', 'CYP2B6', 'CYP2C19', 'CYP2C9', 'CYP2D6', 'CYP3A5', 'DPYD', 'G6PD', 'HLA-A', 'HLA-B', 'MT-RNR1', 'NAT2', 'NUDT15', 'RYR1', 'SLCO1B1', 'TPMT', 'UGT1A1']


Explore the full table — sample a few rows and check the action distribution.

In [30]:
show(df_recs_full.sample(8, random_state=42).to_string(index=False),
     "Sample rows from full CPIC recommendations")

action_counts = df_recs_full["action"].value_counts()
print("\nAction category distribution:")
print(action_counts.to_string())


Action category distribution:
action
No action needed     202
Avoid                199
Alter dose           180
No recommendation     50


### 7.2. On-demand diplotype lookups

There are 110,000+ diplotype entries across all genes, so we fetch them **per gene on demand** and cache the results. The cache means each gene is fetched only once, even across multiple workflow runs.

In [31]:
_diplotype_cache: dict[str, dict[str, str]] = {}

def fetch_diplotypes_for_gene(gene: str) -> dict[str, str]:
    """Return {diplotype: phenotype} for a gene (lazy-fetched and cached)."""
    if gene in _diplotype_cache:
        return _diplotype_cache[gene]
    rows = fetch_all_pages(
        "diplotype",
        select="diplotype,generesult",
        genesymbol=f"eq.{gene}",
    )
    mapping = {r["diplotype"]: r["generesult"] for r in rows}
    _diplotype_cache[gene] = mapping
    print(f"  Cached {len(mapping)} {gene} diplotypes")
    return mapping


def lookup_phenotype_full(gene: str, diplotype: str) -> str | None:
    """Look up a phenotype using the full CPIC API (lazy per-gene fetch)."""
    mapping = fetch_diplotypes_for_gene(gene)
    key = diplotype.strip()
    if key in mapping:
        return mapping[key]
        
    # Try reversed allele order (e.g. *4/*1 → *1/*4)
    parts = key.split("/")
    if len(parts) == 2:
        return mapping.get(f"{parts[1]}/{parts[0]}")
    return None


def lookup_cpic_guideline_full(gene: str, drug: str, phenotype: str) -> dict | None:
    """Look up a CPIC guideline from the full DataFrame (same interface as mini-dataset)."""
    mask = (
        (df_recs_full["gene"] == gene)
        & (df_recs_full["drug"].str.lower() == drug.lower())
        & (df_recs_full["phenotype"] == phenotype)
    )
    matches = df_recs_full[mask]
    if matches.empty:
        return None
    row = matches.iloc[0]
    return {
        "implication":    row["implication"],
        "recommendation": row["recommendation"],
        "classification": row["action"],        # derived action category
        "strength":       row["strength"],      # CPIC evidence strength
    }

In [32]:
# Verify the CYP2D6 *4/*4 example still resolves correctly via the full DB
gene_t, dip_t, drug_t = "CYP2D6", "*4/*4", "codeine"
phen_t  = lookup_phenotype_full(gene_t, dip_t)
guide_t = lookup_cpic_guideline_full(gene_t, drug_t, phen_t)
show(format_guideline(gene_t, dip_t, drug_t, phen_t, guide_t),
     "Full-DB lookup — CYP2D6 *4/*4 + codeine (should match Section 2 mini-dataset)")

  Cached 12700 CYP2D6 diplotypes


### 7.3. Run the workflow against the full database

The helper `run_pgx_workflow_full_db` is a thin wrapper around the LangChain and SDK functions from Sections 3 and 4. The only difference is that it calls `lookup_phenotype_full` and `lookup_cpic_guideline_full` instead of the mini-dataset versions — the LLM interaction logic is completely unchanged.

In [33]:
def run_pgx_workflow_full_db(
    gene: str,
    diplotype: str,
    drug: str,
    generation_model: str = GENERATION_MODEL,
    reflection_model: str = REFLECTION_MODEL,
    use_langchain: bool = True,
) -> dict:
    """
    End-to-end PGx reflection workflow using the full CPIC API database.

    Identical logic to run_pgx_workflow_langchain / run_pgx_workflow_sdk from
    Sections 3 and 4 — only the lookup functions differ.
    """
    impl = "LangChain" if use_langchain else "Direct SDK"
    show(f"Gene: {gene}  |  Diplotype: {diplotype}  |  Drug: {drug}  [{impl}]",
         "Query — Full CPIC DB")

    # Step 1: Generate V1
    show("Step 1: Generating initial recommendation (V1)…")
    if use_langchain:
        v1 = generate_v1_langchain(gene, diplotype, drug, generation_model)
    else:
        v1 = generate_v1_sdk(gene, diplotype, drug, generation_model)
    show(v1, f"Step 1 — V1 [{generation_model}]")

    # Step 2: CPIC lookup (full database)
    show("Step 2: Looking up full CPIC guideline…")
    phenotype = lookup_phenotype_full(gene, diplotype)
    guideline  = lookup_cpic_guideline_full(gene, drug, phenotype) if phenotype else None

    if phenotype is None:
        show(f"Diplotype {diplotype!r} not found in CPIC for {gene}.", "Step 2")
        return {"v1": v1, "v2": v1, "feedback": "No phenotype found."}
    if guideline is None:
        show(f"No CPIC guideline found for {gene} × {drug} × {phenotype}.", "Step 2")
        return {"v1": v1, "v2": v1, "feedback": "No guideline found."}

    show(format_guideline(gene, diplotype, drug, phenotype, guideline),
         "Step 2 — CPIC Guideline (Full DB)")

    # Step 3: Reflect and refine
    show("Step 3: Reflecting on V1 against CPIC guideline…")
    if use_langchain:
        feedback, v2, cls = reflect_and_refine_langchain(
            gene, diplotype, drug, phenotype, v1, guideline, reflection_model,
        )
    else:
        feedback, v2, cls = reflect_and_refine_sdk(
            gene, diplotype, drug, phenotype, v1, guideline, reflection_model,
        )
    show(feedback, "Step 3 — Reflection Feedback")
    show(v2,       f"Step 3 — Refined Recommendation V2 [{reflection_model}]")
    show(f"CPIC Action: {cls}  |  Evidence Strength: {guideline.get('strength', 'N/A')}",
         "Final Classification")

    return {
        "gene": gene, "diplotype": diplotype, "drug": drug,
        "phenotype": phenotype, "v1": v1,
        "guideline": guideline, "feedback": feedback,
        "v2": v2, "classification": cls,
    }

Run the workflow on **CYP2C9 \*3/\*3 + warfarin** — a gene–drug pair not included in the mini-dataset. CYP2C9 encodes a warfarin-metabolizing enzyme; two no-function alleles (*3/*3) markedly reduce warfarin clearance, requiring large dose reductions to avoid bleeding.

In [34]:
result_warfarin = run_pgx_workflow_full_db(
    gene="CYP2C9",
    diplotype="*3/*3",
    drug="warfarin",
    use_langchain=True,
)

  Cached 2850 CYP2C9 diplotypes


Try another gene–drug pair not in the mini-dataset — **SLCO1B1 \*5/\*5 + simvastatin**. SLCO1B1 encodes a hepatic transporter; reduced function leads to elevated plasma statin concentrations and increased risk of myopathy.

In [39]:
result_statin = run_pgx_workflow_full_db(
    gene="SLCO1B1",
    diplotype="*5/*5",
    drug="simvastatin",
    use_langchain=False,    # use direct SDK this time
)

### 7.4. Batch evaluation across multiple gene–drug pairs

The cell below runs V1 generation for a set of six cases, performs the CPIC lookup, and builds a summary DataFrame that shows how much the recommendation changes between V1 and V2. This mirrors the evaluation methodology in the [PGxQA benchmark](https://github.com/KarlKeat/PGxQA/blob/main/scripts/question_generation/diplotype_to_guideline_tests.py), which auto-generates test cases from the full CPIC recommendation table.
> **Note:** This cell calls the LLM six times for V1 generation. Run time is typically 1–2 minutes with `claude-haiku-4-5-20251001`.

In [ ]:
BATCH_EXAMPLES = [
    # gene,        diplotype,       drug,          notes
    ("CYP2D6",  "*4/*4",          "codeine",      "Poor Metabolizer — baseline"),
    ("CYP2D6",  "*1x2/*41x3",     "codeine",      "Ultrarapid Metabolizer — toxicity risk"),
    ("CYP2C9",  "*3/*3",          "warfarin",     "Poor Metabolizer — bleeding risk"),
    ("CYP2C19", "*1/*17",         "clopidogrel",  "Rapid Metabolizer — no action expected"),
    ("SLCO1B1", "*5/*5",          "simvastatin",  "Poor Function — myopathy risk"),
    ("TPMT",    "*3A/*3A",        "azathioprine", "Poor Metabolizer — severe toxicity risk"),
]

batch_results = []
for gene, diplotype, drug, notes in BATCH_EXAMPLES:
    print(f"\n{'='*60}")
    print(f"  {gene} {diplotype} + {drug}")

    # Step 1: V1
    v1 = generate_v1_langchain(gene, diplotype, drug, GENERATION_MODEL)

    # Step 2: CPIC lookup
    phenotype = lookup_phenotype_full(gene, diplotype)
    guideline  = lookup_cpic_guideline_full(gene, drug, phenotype) if phenotype else None

    if phenotype is None or guideline is None:
        print(f"  ⚠ No CPIC data found — skipping reflection")
        batch_results.append({
            "gene": gene, "diplotype": diplotype, "drug": drug,
            "phenotype": "(not found)", "cpic_action": "N/A",
            "v1_first_sentence": v1.split(".")[0],
            "v2_first_sentence": "(skipped)",
            "notes": notes,
        })
        continue

    # Step 3: Reflect
    feedback, v2, cls = reflect_and_refine_langchain(
        gene, diplotype, drug, phenotype, v1, guideline, REFLECTION_MODEL,
    )

    batch_results.append({
        "gene":             gene,
        "diplotype":        diplotype,
        "drug":             drug,
        "phenotype":        phenotype,
        "cpic_action":      guideline["classification"],
        "v1_first_sentence":v1.split(".")[0] + ".",
        "v2_first_sentence":v2.split(".")[0] + ".",
        "notes":            notes,
    })
    print(f"  ✓  phenotype={phenotype}, action={guideline['classification']}")

df_batch = pd.DataFrame(batch_results)
print("\n\nBatch complete — summary table:")
show(df_batch[["gene","drug","phenotype","cpic_action","notes"]].to_string(index=False),
     "Batch Evaluation Summary")

### 7.5. Mini-dataset vs. full database — what changed?

| | **Mini-dataset (Section 2)** | **Full CPIC DB (Section 7)** |
|---|---|---|
| **Coverage** | 3 gene–drug pairs | 17 genes, 50+ drugs, 2,159 recommendations |
| **Diplotype resolution** | Hardcoded lookup dict | Live API, 110k+ entries, lazy-cached per gene |
| **Classification** | Hand-curated labels | Derived from recommendation text and API boolean flags |
| **Data freshness** | Frozen snapshot | Always reflects the current CPIC release |
| **Requires network** | No | Yes (CPIC REST API) |
| **Setup** | None | `pip install requests pandas` (already in environment) |

The **workflow logic** (V1 → lookup → reflect → V2) is completely unchanged. Only the two lookup functions differ, which demonstrates a key design principle: **keep the LLM orchestration decoupled from the data layer** so the data source can be swapped without touching the agent logic.